# 03 Fractional OU Estimation

Fit the fractional OU model to all retained formation spreads. The next module applies anti-persistence and numerical-stability eligibility rules.

Run cells from top to bottom, or choose **Run All** for this notebook only. Each module saves its outputs for the next notebook. Restart the kernel after pulling code changes.


## 1. Imports and saved run


In [ ]:
%matplotlib inline
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'research_config.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open Jupyter inside the Pairs_trading repository.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_session import NotebookSession

session = NotebookSession.active()
cfg = session.config
RUN_DIR = session.run
session.begin('03_fou', ['02_pairs'])


## 2. Load selected formation spreads

No out-of-sample prices enter estimation.


In [ ]:
selected = session.frame('cointegrated_pairs')
formation_spreads = session.frame('formation_spreads')
spreads = {(r.dependent, r.independent): formation_spreads[r.pair] for r in selected.itertuples()}
display(selected[['pair', 'alpha', 'beta']].head())


## 3. Estimate parameters

The continuous-time stationary-variance estimate and daily Euler simulator remain model approximations. Fit errors and boundary H estimates are recorded.


In [ ]:
from src.fractional_OU import fit_cointegrated_pairs_fractional_ou
fou, fit_audit = fit_cointegrated_pairs_fractional_ou(spreads, selected, return_audit=True)
session.save('fou_fit_audit', fit_audit)
if fou.empty:
    raise ValueError('No valid fOU fits. Inspect fou_fit_audit.')
session.save('fractional_ou_parameters', fou)
display(fou.head(10))
display(fit_audit.head(10))


## 4. Hurst summary

The summary refers to exactly the fitted population saved above.


In [ ]:
hurst_summary = fou.hurst.describe()
session.save('hurst_summary', hurst_summary.to_frame('hurst'))
display(hurst_summary)
fou.hurst.hist(bins=25, figsize=(8, 3))
plt.title('Formation Hurst estimates')
plt.xlabel('H')
plt.show()


## Save module completion

Wait for this confirmation before moving to the next notebook.


In [ ]:
session.finish('03_fou')
